# Lekce 07 - Moduly a organizace kódu

Obsah:

* proč organizovat kód
* `*args` a `**kwargs`
* modul je soubor `.py`
* vlastní modul
* skript, nebo modul: `if __name__ == "__main__":`
* past v notebooku: starý modul v paměti
* balíček
* třídy z modulů
* standardní knihovna
* projekt s uv
* jak uspořádat semestrální práci
* semestrální práce
* časté chyby

Na konec, mimo látku: ukázka výkresu DXF z Pythonu.

## 1. Proč organizovat kód

V lekci 6 jsme opakovaný výpočet schovali do funkce. Funkce ale žije jen
v notebooku, kde jste ji napsali. Potřebujete ji v dalším notebooku? Zkopírujete
ji. Najdete v ní chybu? Opravujete ji ve všech kopiích. To je stejný problém,
jaký funkce řešily, jen o patro výš.

Kód se proto organizuje v několika úrovních:

| Úroveň | Co to je | Příklad |
|---|---|---|
| výraz | jeden výpočet | `d / lam` |
| funkce | pojmenovaný výpočet | `odpor_vrstvy(d, lam)` |
| modul | soubor `.py` s funkcemi | `tepelna.py` |
| balíček | složka s moduly | `stavba/` |
| projekt | balíčky, skripty, notebooky a seznam použitých knihoven | `pyproject.toml` |

Dnes projdeme všechny úrovně od funkce nahoru. Začneme ale tím, co lekce 6
nestihla.

## 2. `*args` a `**kwargs`

Funkce `print()` bere libovolný počet argumentů. Jak se taková funkce napíše?
Hvězdička před parametrem posbírá všechny poziční argumenty do jednoho tuple:

In [1]:
def celkovy_odpor(*odpory):
    print(type(odpory), odpory)
    return sum(odpory)


print(celkovy_odpor(1.5, 4.25, 0.25))
print(celkovy_odpor())

<class 'tuple'> (1.5, 4.25, 0.25)
6.0
<class 'tuple'> ()
0


Jméno `args` je jen zvyk, důležitá je hvězdička. Dvě hvězdičky posbírají
pojmenované argumenty do slovníku:

In [2]:
def popis_prvku(kod, **vlastnosti):
    print(kod)
    for klic, hodnota in vlastnosti.items():
        print(f"  {klic}: {hodnota}")


popis_prvku("SO-01", material="C30/37", objem=31.6, podlazi=2)

SO-01
  material: C30/37
  objem: 31.6
  podlazi: 2


Hvězdičky fungují i obráceně, **při volání**. Rozbalí seznam na poziční
argumenty a slovník na pojmenované:

In [3]:
def soucinitel_prostupu(R_konstrukce, R_si=0.13, R_se=0.04):
    return 1 / (R_si + R_konstrukce + R_se)


odpory = [1.5, 4.25, 0.25]
print(celkovy_odpor(*odpory))          # jako celkovy_odpor(1.5, 4.25, 0.25)

okrajove = {"R_si": 0.10, "R_se": 0.04}
print(soucinitel_prostupu(5.90, **okrajove))   # jako R_si=0.10, R_se=0.04

<class 'tuple'> (1.5, 4.25, 0.25)
6.0
0.16556291390728478


Nejčastější použití `**kwargs` je **předání dál**. Funkce nemusí vyjmenovat
všechny parametry funkce, kterou volá. Co nezná, pošle beze změny:

In [4]:
def vypis_U(nazev, R_konstrukce, **kwargs):
    U = soucinitel_prostupu(R_konstrukce, **kwargs)
    print(f"{nazev}: U = {U:.3f} W/(m2.K)")


vypis_U("stena", 5.90)
vypis_U("strop pod pudou", 5.90, R_se=0.10)

stena: U = 0.165 W/(m2.K)
strop pod pudou: U = 0.163 W/(m2.K)


Teď už rozumíte i nápovědě k `print()`. Podívejte se na první řádek
signatury:

In [5]:
help(print)

Help on built-in function print in module builtins:

print(*args, sep=' ', end='\n', file=None, flush=False)
    Prints the values to a stream, or to sys.stdout by default.

    sep
      string inserted between values, default a space.
    end
      string appended after the last value, default a newline.
    file
      a file-like object (stream); defaults to the current sys.stdout.
    flush
      whether to forcibly flush the stream.



## 3. Modul je soubor `.py`

**Modul** je obyčejný soubor s kódem v Pythonu. `math`, který používáme od
lekce 2, je modul ze standardní knihovny. Příkaz `import` soubor najde, spustí
a výsledek uloží pod jméno modulu:

In [6]:
import math

print(type(math))
print(math.sqrt(2), math.pi)

<class 'module'>
1.4142135623730951 3.141592653589793


Modul je **jmenný prostor**. Jména uvnitř modulu se nepletou s vašimi, stejně
jako se lokální jména funkce nepletla s globálními (lekce 6). `math.pi` a vaše
vlastní `pi` jsou dvě různé věci. Co modul obsahuje, ukáže `dir()`:

In [7]:
pi = 3        # vlastni pi, math.pi tim nezmenime
print(pi, math.pi)

print([jmeno for jmeno in dir(math) if not jmeno.startswith("_")][:12])

3 3.141592653589793
['acos', 'acosh', 'asin', 'asinh', 'atan', 'atan2', 'atanh', 'cbrt', 'ceil', 'comb', 'copysign', 'cos']


Import má tři běžné podoby:

| Zápis | Použití v kódu | Kdy |
|---|---|---|
| `import math` | `math.sqrt(2)` | výchozí volba, hned je vidět, odkud jméno je |
| `from math import sqrt, pi` | `sqrt(2)` | pár jmen, která používáte často |
| `import statistics as st` | `st.mean(data)` | dlouhé jméno modulu nebo zvyklost (`import numpy as np`) |

Čtvrtou podobu, `from math import *`, nepoužívejte. Natáhne do vašeho jmenného
prostoru všechno a potichu přepíše jména, která už máte:

In [8]:
e = 0.05            # excentricita sily v m
from math import *
print(e)            # nase e je pryc, je tu Eulerovo cislo

2.718281828459045


## 4. Vlastní modul

Funkce z lekce 6 přesuneme do souboru `tepelna.py`. Ve cvičení soubor vytvoříte
ve VS Code. V přednášce ho zapíšeme přímo z notebooku buňkou s `%%writefile`,
abychom jeho obsah viděli. Výsledek je stejný: soubor vedle notebooku.

Na začátku souboru je **docstring modulu**. Popisuje, k čemu modul je.

In [9]:
%%writefile tepelna.py
"""Vypocty tepelne techniky pro skladbu konstrukce."""


def odpor_vrstvy(d: float, lam: float) -> float:
    """Vrati tepelny odpor vrstvy v m2.K/W."""
    return d / lam


def celkovy_odpor(*odpory: float) -> float:
    """Secte odpory libovolneho poctu vrstev."""
    return sum(odpory)


def soucinitel_prostupu(R_konstrukce: float, R_si: float = 0.13,
                        R_se: float = 0.04) -> float:
    """Vrati soucinitel prostupu tepla U ve W/(m2.K)."""
    return 1 / (R_si + R_konstrukce + R_se)

Writing tepelna.py


In [10]:
import tepelna

R = tepelna.celkovy_odpor(
    tepelna.odpor_vrstvy(0.30, 0.18),
    tepelna.odpor_vrstvy(0.16, 0.038),
    tepelna.odpor_vrstvy(0.015, 0.80),
)
print(f"R = {R:.2f} m2.K/W, U = {tepelna.soucinitel_prostupu(R):.3f} W/(m2.K)")

R = 5.90 m2.K/W, U = 0.165 W/(m2.K)


Docstringy modulu i funkcí teď vypíše `help()`:

In [11]:
help(tepelna)

Help on module tepelna:

NAME
    tepelna - Vypocty tepelne techniky pro skladbu konstrukce.

FUNCTIONS
    celkovy_odpor(*odpory: float) -> float
        Secte odpory libovolneho poctu vrstev.

    odpor_vrstvy(d: float, lam: float) -> float
        Vrati tepelny odpor vrstvy v m2.K/W.

    soucinitel_prostupu(R_konstrukce: float, R_si: float = 0.13, R_se: float = 0.04) -> float
        Vrati soucinitel prostupu tepla U ve W/(m2.K).

FILE
    tepelna.py




**Kde Python moduly hledá?** Postupně ve:

1. složce se spuštěným skriptem, u notebooku ve složce s notebookem
2. standardní knihovně
3. balíčcích nainstalovaných v aktivním prostředí (`.venv`)

Seznam těchto míst je v `sys.path`. Pro vlastní moduly z toho plyne jednoduché
pravidlo: soubor patří do stejné složky jako notebook nebo skript, který ho
importuje.

## 5. Skript, nebo modul: `if __name__ == "__main__":`

`import` soubor **spustí**. Všechno, co je v něm mimo funkce, se při importu
provede. Vyzkoušíme to na malém souboru:

In [12]:
%%writefile pozdrav.py
print("Spoustim soubor pozdrav.py, __name__ =", __name__)


def ahoj(jmeno):
    return f"Ahoj, {jmeno}!"

Writing pozdrav.py


In [13]:
import pozdrav

print(pozdrav.ahoj("Jano"))

Spoustim soubor pozdrav.py, __name__ = pozdrav
Ahoj, Jano!


Druhý `import` už nic nevypíše. Python si modul pamatuje a podruhé ho
nespouští:

In [14]:
import pozdrav

Proměnná `__name__` obsahuje jméno modulu. Když ale soubor spustíte přímo,
v terminálu příkazem `uv run pozdrav.py`, dostane jméno `"__main__"`.
V notebooku totéž udělá `%run`:

In [15]:
%run pozdrav.py

Spoustim soubor pozdrav.py, __name__ = __main__


Toho se využívá k rozlišení dvou rolí téhož souboru:

```python
def ahoj(jmeno):
    return f"Ahoj, {jmeno}!"


if __name__ == "__main__":
    # spusti se jen pri "uv run pozdrav.py", ne pri importu
    print(ahoj("svete"))
```

Pod `if __name__ == "__main__":` patří ukázka, rychlý test nebo hlavní výpočet
skriptu. Kdo modul importuje, chce jen jeho funkce, ne výpisy.

## 6. Past v notebooku: starý modul v paměti

Do `tepelna.py` přidáme funkci `potrebna_tloustka` a blok `__main__`:

In [16]:
%%writefile tepelna.py
"""Vypocty tepelne techniky pro skladbu konstrukce."""


def odpor_vrstvy(d: float, lam: float) -> float:
    """Vrati tepelny odpor vrstvy v m2.K/W."""
    return d / lam


def celkovy_odpor(*odpory: float) -> float:
    """Secte odpory libovolneho poctu vrstev."""
    return sum(odpory)


def soucinitel_prostupu(R_konstrukce: float, R_si: float = 0.13,
                        R_se: float = 0.04) -> float:
    """Vrati soucinitel prostupu tepla U ve W/(m2.K)."""
    return 1 / (R_si + R_konstrukce + R_se)


def potrebna_tloustka(U_pozadovane: float, R_ostatni: float, lam: float,
                      R_si: float = 0.13, R_se: float = 0.04) -> float:
    """Vrati tloustku izolace v m, se kterou konstrukce splni U_pozadovane."""
    R_potrebne = 1 / U_pozadovane - R_si - R_se - R_ostatni
    return R_potrebne * lam


if __name__ == "__main__":
    R = celkovy_odpor(odpor_vrstvy(0.30, 0.18), odpor_vrstvy(0.16, 0.038))
    print(f"Ukazka: R = {R:.2f} m2.K/W, U = {soucinitel_prostupu(R):.3f} W/(m2.K)")

Overwriting tepelna.py


Soubor je uložený, nová funkce v něm je. A přesto:

In [17]:
import tepelna

R_zdivo = tepelna.odpor_vrstvy(0.30, 0.18)
print(tepelna.potrebna_tloustka(0.25, R_zdivo, 0.038))

AttributeError: module 'tepelna' has no attribute 'potrebna_tloustka'

Python modul načetl už v oddílu 4 a nový `import` ho znovu nečte (viz
`pozdrav.py`). Notebook pořád používá starou verzi. Možnosti nápravy:

* **restart kernelu** - spolehlivé, ale přijdete o všechny proměnné
* **`importlib.reload(modul)`** - znovu načte jeden modul
* **`%autoreload`** - notebook před každou buňkou sám načte změněné moduly;
  na začátek notebooku dejte:

```python
%load_ext autoreload
%autoreload 2
```

In [18]:
import importlib

importlib.reload(tepelna)

d = tepelna.potrebna_tloustka(0.25, R_zdivo, 0.038)
print(f"Pro U = 0,25 W/(m2.K) je potreba {d * 100:.1f} cm izolace")

Pro U = 0,25 W/(m2.K) je potreba 8.2 cm izolace

Všimněte si, že `reload` modul znovu spustil, ale ukázka z bloku `__main__`
se nevypsala. Při spuštění souboru jako skriptu se vypíše:

In [19]:
%run tepelna.py

Ukazka: R = 5.88 m2.K/W, U = 0.165 W/(m2.K)


## 7. Balíček

Když modulů přibývá, sdružíme je do **balíčku**. Balíček je složka s moduly
a se souborem `__init__.py`, který z ní balíček dělá. `__init__.py` může být
prázdný nebo obsahovat jen docstring balíčku.

```
lectures/
    Lecture7.ipynb
    stavba/
        __init__.py
        geometrie.py
```

Do balíčku `stavba` dáme výpočty z lekce 5: obvod a plochu půdorysu zadaného
seznamem bodů.

In [20]:
import os

os.makedirs("stavba", exist_ok=True)     # slozku muzete vytvorit i ve VS Code

In [21]:
%%writefile stavba/__init__.py
"""Vypocty pro stavebni konstrukce."""

Writing stavba/__init__.py


In [22]:
%%writefile stavba/geometrie.py
"""Geometrie pudorysu zadaneho seznamem bodu [(x, y), ...]."""

import math


def obvod(body: list) -> float:
    """Vrati obvod uzavreneho polygonu v m."""
    return sum([math.dist(a, b) for a, b in zip(body, body[1:] + body[:1])])


def plocha(body: list) -> float:
    """Vrati plochu polygonu v m2 (vzorec tkanicky)."""
    s = sum([a[0] * b[1] - b[0] * a[1] for a, b in zip(body, body[1:] + body[:1])])
    return abs(s) / 2

Writing stavba/geometrie.py


Modul z balíčku se adresuje tečkou: `stavba.geometrie`. Opět funguje
`import` i `from ... import`:

In [23]:
from stavba import geometrie
from stavba.geometrie import plocha

pudorys = [(0.0, 0.0), (12.0, 0.0), (12.0, 8.0), (5.0, 8.0), (5.0, 5.0), (0.0, 5.0)]

print(f"Obvod {geometrie.obvod(pudorys):.1f} m, plocha {plocha(pudorys):.1f} m2")

Obvod 40.0 m, plocha 81.0 m2


Pro semestrální práci stačí přesně tohle: jedna složka s několika moduly.
Balíček v balíčku nebo zveřejnění na PyPI potřebovat nebudete.

## 8. Třídy z modulů

Moduly neobsahují jen funkce a konstanty, ale i **třídy**. Třída je typ, podle
kterého se vytvářejí objekty. Zavolání třídy vytvoří nový objekt a ten má
metody (lekce 6). V lekci 3 jste si takovou třídu sami nadefinovali pomocí
`dataclass`. Teď jen použijeme hotové třídy z modulů:

In [24]:
from datetime import date, timedelta

betonaz = date(2026, 10, 19)                # trida date vytvori objekt
odbedneni = betonaz + timedelta(days=28)

print(type(betonaz))
print("Odbedneni:", odbedneni, "den v tydnu:", odbedneni.weekday())   # 0 = pondeli

<class 'datetime.date'>
Odbedneni: 2026-11-16 den v tydnu: 0


In [25]:
from collections import Counter

materialy = ["C25/30", "C30/37", "C25/30", "C20/25", "C25/30"]
pocty = Counter(materialy)

print(pocty)
print(pocty.most_common(1))

Counter({'C25/30': 3, 'C30/37': 1, 'C20/25': 1})
[('C25/30', 3)]


Vlastní třídy v tomto kurzu psát nebudeme, používat je ale budete pořád.
Jednotky z balíčku `pint`, výkres z `ezdxf`, pole z NumPy nebo tabulka z Pandas
jsou objekty nějaké třídy. Pracuje se s nimi stejně: vytvořit objekt, volat
jeho metody.

## 9. Standardní knihovna

Python se dodává se stovkami modulů, kterým se říká **standardní knihovna**.
Nic se neinstaluje, stačí `import`. Několik užitečných:

| Modul | K čemu |
|---|---|
| `math` | matematické funkce a konstanty |
| `statistics` | průměr, medián, směrodatná odchylka |
| `random` | náhodná čísla a náhodný výběr |
| `datetime` | data a časy |
| `itertools` | kombinace, permutace a další iterace |
| `collections` | `Counter` a další kolekce |
| `copy` | kopie objektů (lekce 3) |
| `pathlib`, `json`, `csv` | práce se soubory (lekce 10) |
| `doctest`, `unittest` | testy (lekce 11) |

Než začnete něco programovat sami, vyplatí se zkontrolovat, jestli to už
standardní knihovna neumí:

In [26]:
import statistics
from itertools import combinations

pevnosti = [31.2, 29.8, 33.5, 30.1, 28.7, 32.4]    # MPa, zkousky krychli
print(f"prumer {statistics.mean(pevnosti):.1f} MPa, "
      f"smerodatna odchylka {statistics.stdev(pevnosti):.2f} MPa")

zatizeni = ["vlastni tiha", "uzitne", "snih", "vitr"]
for dvojice in combinations(zatizeni, 2):
    print(dvojice)

prumer 30.9 MPa, smerodatna odchylka 1.78 MPa
('vlastni tiha', 'uzitne')
('vlastni tiha', 'snih')
('vlastni tiha', 'vitr')
('uzitne', 'snih')
('uzitne', 'vitr')
('snih', 'vitr')


## 10. Projekt s uv

Ve cvičení 1 jste prostředí vytvořili ručně příkazy `uv venv`
a `uv pip install ipykernel`. Na jednom počítači to stačí. Jenže `.venv` si
nikde nezapisuje, co jste do něj nainstalovali, a kolega podle ničeho stejné
prostředí nesestaví. Vy sami za půl roku také ne. Proto má uv **projekty**.

Nový projekt založíte v terminálu:

```
uv init --no-package izolace
cd izolace
uv add pint
uv add --dev ipykernel
uv run main.py
```

Přepínač `--no-package` dá jednoduchou strukturu se souborem `main.py`. Bez něj
novější uv připraví složku `src/` pro balíček určený ke zveřejnění, což pro
semestrální práci nepotřebujeme.

Po těchto příkazech vypadá projekt takto:

```
izolace/
    .git/              # uv rovnou zalozi git repozitar, k nemu se dostaneme v lekci 12
    .gitignore
    .python-version    # verze Pythonu pro projekt
    .venv/             # prostredi, uv ho vytvori sam
    main.py
    pyproject.toml     # popis projektu a seznam zavislosti
    README.md
    uv.lock            # presne verze vsech nainstalovanych balicku
```

Soubor `pyproject.toml` po `uv add`:

```toml
[project]
name = "izolace"
version = "0.1.0"
description = "Add your description here"
readme = "README.md"
requires-python = ">=3.13"
dependencies = [
    "pint>=0.26.1",
]

[dependency-groups]
dev = [
    "ipykernel>=7.4.0",
]
```

A `main.py`, který uv vytvoří, už znáte z oddílu 5:

```python
def main():
    print("Hello from izolace!")


if __name__ == "__main__":
    main()
```

Základní příkazy:

| Příkaz | Co udělá |
|---|---|
| `uv init --no-package nazev` | založí projekt ve složce `nazev` |
| `uv add balicek` | nainstaluje balíček a zapíše ho do `pyproject.toml` a `uv.lock` |
| `uv add --dev balicek` | totéž pro nástroje, které potřebujete jen při vývoji, např. `ipykernel` |
| `uv remove balicek` | balíček z projektu odebere |
| `uv sync` | sestaví `.venv` přesně podle `uv.lock` |
| `uv run skript.py` | spustí skript v prostředí projektu; když je potřeba, nejdřív ho sesynchronizuje |

Rozdíl proti cvičení 1: `uv pip install` balíček jen nainstaluje do `.venv`.
`uv add` si ho navíc zapíše, takže prostředí jde kdykoli sestavit znovu.

**Co si schovat a co ne:**

* `pyproject.toml` a `uv.lock` jsou recept na prostředí. Patří k projektu
  a posílají se spolu s kódem.
* `.venv` je jen výsledek receptu. Můžete ho kdykoli smazat a `uv sync` ho
  postaví znovu. Neposílá se, neodevzdává se a do gitu nepatří.

**Notebook v projektu:** ve VS Code vyberte vpravo nahoře kernel z `.venv`
projektu (Select Kernel, Python Environments, `.venv`). Díky
`uv add --dev ipykernel` v něm notebook poběží.

## 11. Jak uspořádat semestrální práci

Doporučená struktura:

```
moje-prace/
    pyproject.toml
    uv.lock
    README.md          # o cem prace je a jak ji spustit
    main.py            # spusti cely vypocet: uv run main.py
    analyza.ipynb      # zkouseni, mezivysledky, grafy
    vypocet/           # balicek s funkcemi
        __init__.py
        zatizeni.py
        prurezy.py
```

Pravidla:

* **funkce do modulů, notebook je jen volá** - co se dá použít dvakrát,
  nepatří do buňky notebooku
* **jeden modul, jedno téma** - `zatizeni.py`, `prurezy.py`, ne `funkce.py`
  se vším
* **jména modulů** malými písmeny s podtržítkem, bez mezer, pomlček
  a diakritiky, a ne stejně jako modul ze standardní knihovny nebo balíček
  (`math.py`, `random.py`, `statistics.py`, `pint.py`)
* **importy na začátek souboru**, v pořadí: standardní knihovna, cizí
  balíčky, vlastní moduly:

```python
import math

import pint

from vypocet import zatizeni
```

## 12. Semestrální práce

*Zadání, témata a termíny doplní vyučující.*

## 13. Časté chyby

**Přípona v importu.** Importuje se jméno modulu, ne jméno souboru:

In [27]:
import tepelna.py

ModuleNotFoundError: No module named 'tepelna.py'; 'tepelna' is not a package

**Modul nenalezen.** Buď je ve jméně překlep, nebo balíček není nainstalovaný
v prostředí, ve kterém běží kernel. Častá příčina: balíček jste přidali do
projektu, ale notebook má vybraný jiný kernel.

In [28]:
import pnit

ModuleNotFoundError: No module named 'pnit'

**Vlastní soubor se jménem modulu ze standardní knihovny.** Když vedle
skriptu leží váš `statistics.py`, `import statistics` načte ten váš. Python 3.13
to při spuštění skriptu rovnou poradí:

```
AttributeError: module 'statistics' has no attribute 'mean' (consider renaming
'...\statistics.py' since it has the same name as the standard library module
named 'statistics' and prevents importing that standard library module)
```

Řešení: soubor přejmenovat. Vyzkoušíte si to ve cvičení.

**Změna v modulu se neprojeví.** Notebook drží starou verzi, viz oddíl 6.

**`from modul import *`** potichu přepíše vaše jména, viz oddíl 3.

**Pomlčka ve jméně souboru.** `moje-funkce.py` nejde importovat, protože
`import moje-funkce` je pro Python odčítání. Použijte `moje_funkce.py`.

## Shrnutí

| Zápis | Význam |
|---|---|
| `def f(*args)` | libovolný počet pozičních argumentů, uvnitř je tuple |
| `def f(**kwargs)` | libovolné pojmenované argumenty, uvnitř je slovník |
| `f(*seznam)`, `f(**slovnik)` | rozbalení při volání |
| `import modul` | použití jako `modul.jmeno` |
| `from modul import jmeno` | použití rovnou jako `jmeno` |
| `import modul as m` | kratší jméno modulu |
| `from balicek.modul import jmeno` | import z balíčku |
| `if __name__ == "__main__":` | kód jen pro spuštění souboru jako skriptu |
| `importlib.reload(modul)`, `%autoreload 2` | načtení změněného modulu v notebooku |
| `uv init --no-package nazev` | nový projekt |
| `uv add balicek` | přidání balíčku do projektu |
| `uv sync` | sestavení `.venv` podle `uv.lock` |
| `uv run skript.py` | spuštění v prostředí projektu |

Co si odnést:

* modul je soubor `.py`, balíček je složka s moduly a `__init__.py`
* modul je jmenný prostor; `import modul` je nejčitelnější, `import *` nikdy
* import soubor spustí, a to jen jednou; kód mimo funkce dejte pod
  `if __name__ == "__main__":`
* v notebooku se změny modulu neprojeví samy, zapněte `%autoreload`
* třídy z modulů se používají stejně jako funkce: zavolání vytvoří objekt,
  objekt má metody
* projekt s uv si pamatuje své balíčky v `pyproject.toml` a `uv.lock`;
  `.venv` je na jedno použití
* funkce patří do modulů, notebook je jen volá

V lekci 8 přidáme do projektu první velké balíčky: NumPy a Matplotlib.

## Ukázka: co Python ještě umí

*Tahle část není látka do testu. Nemusíte rozumět každému řádku.*

Balíček `ezdxf` zapisuje výkresy ve formátu DXF, který otevře AutoCAD i další
CAD programy. Spojíme ho s vlastním balíčkem `stavba` z oddílu 7: půdorys se
spočítá i nakreslí jedním skriptem. Instalace: `uv add ezdxf` v projektu, nebo
`uv pip install ezdxf`.

In [29]:
import ezdxf

from stavba.geometrie import obvod, plocha

pudorys = [(0.0, 0.0), (12.0, 0.0), (12.0, 8.0), (5.0, 8.0), (5.0, 5.0), (0.0, 5.0)]

vykres = ezdxf.new(setup=True)            # novy vykres s prednastavenymi styly
msp = vykres.modelspace()                 # modelovy prostor, sem se kresli
vykres.layers.add("OBRYS", color=1)
vykres.layers.add("KOTY", color=3)

msp.add_lwpolyline(pudorys, close=True, dxfattribs={"layer": "OBRYS"})

# kota pod spodni hranou a vpravo od prave hrany; kreslime v m, kotujeme v mm
v_mm = {"dimlfac": 1000}
msp.add_linear_dim(base=(0, -1.5), p1=(0, 0), p2=(12, 0), override=v_mm,
                   dxfattribs={"layer": "KOTY"}).render()
msp.add_linear_dim(base=(13.5, 0), p1=(12, 0), p2=(12, 8), angle=90, override=v_mm,
                   dxfattribs={"layer": "KOTY"}).render()

popis = f"plocha {plocha(pudorys):.1f} m2, obvod {obvod(pudorys):.1f} m"
msp.add_text(popis, height=0.4).set_placement((1.0, 2.0))

vykres.saveas("pudorys.dxf")
print("Ulozeno: pudorys.dxf,", len(msp), "entit")

Ulozeno: pudorys.dxf, 4 entit


Soubor `pudorys.dxf` otevřete v AutoCADu, v LibreCADu nebo v libovolném
online prohlížeči DXF:

![Půdorys vygenerovaný balíčkem ezdxf](Figures/pudorys.png)

Stejně se dají generovat výkresy výztuže, řezy nebo výkazy. Změna jednoho čísla
ve skriptu překreslí celý výkres.